# Week 4 Breakout Slides: Unsupervised Learning

In the Data Exploration notebook we measured how country-level indicators relate and put them on a common scale. Here we take the next step: letting the data organize *itself*. **Unsupervised learning** finds structure without any labels to guide it. We use **PCA** to compress many correlated indicators into a handful of informative directions, then **K-Means** to group similar countries into clusters. Throughout we rely on `scikit-learn`, the standard machine-learning library for Python, so we never have to write these algorithms by hand.

## Import Libraries

We start with the familiar trio: `numpy`, `pandas`, and `matplotlib`. The `scikit-learn` pieces (the scaler, PCA, and K-Means) are imported later, each right before we use it, so it's clear which tool does what.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

## Load and Inspect the Data

We reuse the same country development dataset from the Data Exploration notebook (one row per country, one column per indicator).

In [ ]:
path_to_file = Path('your/path')

In [ ]:
df = pd.read_csv(path_to_file / 'country_data.csv')

### Inspect the DataFrame

A quick look confirms the columns and shape before we start modeling.

In [ ]:
df

## Standardize the Features

Both PCA and K-Means are driven by distances and variances, so features on larger scales would dominate the analysis. We put every column on a common scale first. `scikit-learn`'s `StandardScaler` computes the z-score (subtract the mean, divide by the standard deviation) the same transformation we did by hand in the Data Exploration notebook, now as a one-liner.

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
df_z = StandardScaler().fit_transform(df)

## Principal Component Analysis (PCA)

Many of our indicators are correlated. Income, GDP per capita, and life expectancy all move together, for instance. **PCA** rotates the data onto a new set of axes, the **principal components**, which are linear combinations of the original features ordered by how much variance they capture. The first few components often summarize most of the information, letting us reduce dimensionality while throwing away very little.

In [ ]:
from sklearn.decomposition import PCA

### Fit PCA

Calling `.fit()` with no `n_components` keeps every component, which is exactly what we want for the diagnostics below — we need to see the full variance profile before deciding how many components to keep.

In [ ]:
model_pca = PCA().fit(df_z)

### Diagnostic: Scree Plot

A **scree plot** shows the proportion of total variance each principal component explains (`explained_variance_ratio_`). Components are sorted from most to least informative, so the curve always descends. We look for the "elbow." The elbow (or "knee") is the point after which additional components add little.

In [ ]:
pcs = np.arange(1, len(model_pca.explained_variance_ratio_) + 1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(pcs, model_pca.explained_variance_ratio_, marker='o', color='steelblue')

ax.set_xticks(pcs)
ax.set_xlabel('Principal Component')
ax.set_ylabel('Proportion of Variance Explained')
ax.set_title('Scree Plot')
ax.grid(alpha=0.3)

### Diagnostic: Cumulative Explained Variance

Summing the ratios as we add components tells us how much total variance a given number of components retains. A common rule of thumb is to keep enough components to explain ~90% of the variance; the dashed line marks that threshold.

In [ ]:
cum_var = np.cumsum(model_pca.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(pcs, cum_var, marker='o', color='steelblue')
ax.axhline(0.9, color='k', linestyle='--', label='90% threshold')

ax.set_xticks(pcs)
ax.set_ylim([0, 1.02])
ax.set_xlabel('Number of Principal Components')
ax.set_ylabel('Cumulative Variance Explained')
ax.set_title('Cumulative Explained Variance')
ax.legend()
ax.grid(alpha=0.3)

### Project the Data onto the Principal Components

`.transform()` rotates the standardized data onto the principal-component axes. We cluster on this representation: the components are uncorrelated and ordered by information content, which makes the downstream K-Means both cleaner and easier to visualize.

In [ ]:
df_z_pc = model_pca.transform(df_z)

### Interpreting the Axes: Component Loadings

A scatter on PC 1 vs. PC 2 is only meaningful if we know *what those axes represent*. Each principal component is a weighted mix of the original indicators, and those weights — the **loadings**, stored in `model_pca.components_` — tell us which features pull a country in which direction. Reading them lets us translate "far right on PC 1" into plain language like "high income, long life expectancy." A large positive loading means the feature pushes a country toward the positive end of that axis; a large negative loading, toward the negative end.

In [ ]:
loadings = pd.DataFrame(model_pca.components_[:2].T,
                        index=df.columns,
                        columns=['PC 1', 'PC 2'])

fig, ax = plt.subplots(figsize=(5, 6))
im = ax.imshow(loadings, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')

cbar = ax.figure.colorbar(im, ax=ax)
cbar.ax.set_ylabel('Loading', rotation=-90, va='bottom')

ax.set_xticks(np.arange(loadings.shape[1]))
ax.set_xticklabels(loadings.columns)
ax.set_yticks(np.arange(loadings.shape[0]))
ax.set_yticklabels(loadings.index)

# annotate each cell with its loading value
for i in range(loadings.shape[0]):
    for j in range(loadings.shape[1]):
        ax.text(j, i, f'{loadings.iloc[i, j]:.2f}', ha='center', va='center', fontsize=9)

ax.set_title('PCA Loadings')
fig.tight_layout()
plt.show()

## K-Means Clustering

**K-Means** partitions the observations into `k` groups by repeatedly assigning each point to its nearest cluster center and then moving each center to the mean of its assigned points. It minimizes the within-cluster sum of squared distances (the **inertia**). The catch: we have to choose `k` ourselves, so we lean on diagnostics to guide that choice.

In [ ]:
from sklearn.cluster import KMeans

### Choosing k: Diagnostics

There's no label to tell us the "right" number of clusters, so we run K-Means across a range of `k` values and inspect two complementary diagnostics.

#### Elbow Method

We fit K-Means for a range of `k` and record the **inertia** each time. Inertia always falls as `k` grows (more centers fit the data more tightly), so we don't just minimize it — we look for the **elbow**, where the rate of improvement sharply flattens. That bend marks the point of diminishing returns.

In [ ]:
ks = range(1, 11)
inertias = [KMeans(n_clusters=k, random_state=42, n_init=10).fit(df_z_pc).inertia_
            for k in ks]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(ks), inertias, marker='o', color='steelblue')

ax.set_xticks(list(ks))
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Inertia (within-cluster sum of squares)')
ax.set_title('Elbow Method')
ax.grid(alpha=0.3)

#### Silhouette Analysis

The **silhouette score** measures how well each point sits inside its own cluster versus the nearest neighboring cluster, averaged over all points. It ranges from -1 to 1, where higher is better and values near 0 signal overlapping clusters. Because it compares at least two clusters, we start the sweep at `k = 2`. The peak suggests a well-separated choice of `k`.

In [ ]:
from sklearn.metrics import silhouette_score

ks = range(2, 11)
sil_scores = [silhouette_score(df_z_pc,
                               KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(df_z_pc))
              for k in ks]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(ks), sil_scores, marker='o', color='steelblue')

ax.set_xticks(list(ks))
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Mean Silhouette Score')
ax.set_title('Silhouette Analysis')
ax.grid(alpha=0.3)

### Fit the Final Model

Guided by the diagnostics, we settle on `k = 4` and fit the final model. Setting `random_state` makes the result reproducible, and `n_init=10` runs the algorithm from several random starts and keeps the best, guarding against poor local minima.

In [ ]:
mod_km = KMeans(n_clusters=4, random_state=42, n_init=10)
mod_km.fit(df_z_pc)

### Cluster Assignments

`.predict()` returns the cluster label (0 to 3) assigned to each country.

In [ ]:
labels = mod_km.predict(df_z_pc)
labels

### Visualize the Clusters

The clearest way to sanity-check the result is to plot the countries on the first two principal components (the axes that carry the most variance) colored by their cluster label. The red markers show each cluster's centroid. Well-formed clusters appear as distinct, coherent groups — and using the loadings above, we can read *where* a cluster sits (left/right, up/down) as a statement about the underlying indicators.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

scatter = ax.scatter(df_z_pc[:, 0], df_z_pc[:, 1],
                     c=labels, cmap='viridis', s=40, alpha=0.8)

# overlay the cluster centroids
ax.scatter(mod_km.cluster_centers_[:, 0], mod_km.cluster_centers_[:, 1],
           c='red', marker='X', s=200, edgecolor='k', label='Centroids')

fig.colorbar(scatter, ax=ax, label='Cluster')

ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.set_title('K-Means Clusters in PCA Space')
ax.legend(loc='best')

### Interpreting the Clusters: Cluster Profiles

The scatter shows *that* the countries separate; to say *what each cluster means*, we go back to the original, unscaled indicators. Attaching the cluster labels to the raw DataFrame and averaging each feature per cluster gives a readable profile — one row per cluster, in the original units (dollars, years, percentages). This is where the analysis pays off: each cluster becomes a recognizable group of countries, such as low-income/high-child-mortality versus wealthy/high-life-expectancy.

In [ ]:
df_profile = df.copy()
df_profile['cluster'] = labels

df_profile.groupby('cluster').mean().round(2)